# 🧬 VERA Master Learning Lab: Complete Self-Contained RAG Pipeline
## 🩺 المختبر التعليمي الشامل لمنظومة الذكاء الاصطناعي السريري (VERA)

---

### 🎯 الهدف من هذه المفكرة (Objective):
تم تصميم هذه المفكرة لتكون **دليلاً تعليمياً وتطبيقياً مستقلاً 100%**، يحتوي على **كود بايثون بسيط ومباشر بدون تعقيدات الكلاسات والملفات المتعددة**.
يمكنك من خلالها فهم وتجربة كل مرحلة من مراحل الـ **RAG (Retrieval-Augmented Generation)** خطوة بخطوة:

1. 📦 **المكتبات المستخدمة**: شرح دور كل مكتبة في معالجة البيانات والذكاء الاصطناعي.
2. 📄 **استخراج النصوص من ملفات PDF**: قراءة المستندات الطبية واستخراج النصوص والصفحات.
3. ✂️ **تقطيع النصوص (Chunking)**: تقسيم المستندات لمقاطع ذكية مع حفظ الميتاداتا (رقم الصفحة واسم الملف).
4. 🧠 **تحويل النصوص لمتجهات (Embeddings)**: تحويل المعاني اللغوية إلى متجهات رقمية.
5. 🗄️ **تخزين وفهرسة المتجهات (ChromaDB + BM25)**: الجمع بين البحث الدلالي وبحث الكلمات المفتاحية.
6. 🔍 **البحث الهجين (Hybrid Retrieval & RRF)**: دمج أفضل النتائج باستخدام Reciprocal Rank Fusion.
7. 🌐 **توسيع الاستعلامات الطبية (Query Expansion)**: دعم المصطلحات الطبية باللغتين العربية والإنجليزية.
8. 🛡️ **بوابات الأمان وتدقيق الهلوسة (Safety Guardrails)**: التأكد من دقة الأدلة وعدم اختلاق معلومات.
9. ✍️ **التوليد السريري المنضبط بالأدلة (Grounded LLM Generation)**: صياغة التوصيات مع الاستشهادات الدقيقة `[Doc | Sec | Page]`.
10. 🧪 **مختبر الاختبار التفاعلي (Interactive End-to-End Tester)**: صندوق تجارب لاختبار أي سؤال طبي مباشرة.

---

## 📦 المرحلة 1: استيراد المكتبات الأساسية وشرح دورها

| المكتبة | دورها في المنظومة |
|:---|:---|
| `pypdf` / `pdfplumber` | استخراج النصوص والجداول والصفحات من ملفات الـ PDF الطبية بدقة. |
| `sentence_transformers` | توليد المتجهات الدلالية (Embeddings) عالية الدقة للنصوص الطبية. |
| `chromadb` | قاعدة بيانات متجهة خفيفة وسريعة لتخزين والبحث في الـ Embeddings. |
| `rank_bm25` | خوارزمية بحث تقليدية تعتمد على الكلمات المفتاحية (Lexical Keyword Search). |
| `google.generativeai` / `openai` | نماذج اللغة الكبيرة (LLMs) لتوليد الإجابات المنضبطة بالأدلة. |

In [ ]:
# 1. استيراد المكتبات الأساسية
import os
import re
import json
import time
from pathlib import Path

# مكتبات قراءة الـ PDF
import pypdf
import pdfplumber

# مكتبات المتجهات وقواعد البيانات
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

print("✅ تم استيراد جميع المكتبات الأساسية بنجاح!")

## 📄 المرحلة 2: قراءة وتحميل ملفات الـ PDF الطبية

في هذه الخطوة سنقرأ الأوراق البحثية الطبية الموجودة في المجلد ونستخرج نصوصها صفحة بصفحة مع حفظ:
- اسم الملف (`filename`)
- رقم الصفحة (`page_number`)
- النص المستخرج (`text`)

In [ ]:
# مسار مجلد ملفات الـ PDF
PDF_DIR = "../data/raw_pdfs"
if not os.path.exists(PDF_DIR):
    PDF_DIR = "./data/raw_pdfs" # في حال تشغيلها من المجلد الرئيسي

# دالة بسيطة لقراءة ملف PDF صفحة بصفحة
def load_pdf_pages(pdf_path):
    reader = pypdf.PdfReader(pdf_path)
    pages_data = []
    filename = os.path.basename(pdf_path)
    
    for idx, page in enumerate(reader.pages, 1):
        text = page.extract_text() or ""
        # تنظيف بسيط للمسافات الزائدة
        cleaned_text = " ".join(text.split())
        if cleaned_text:
            pages_data.append({
                "filename": filename,
                "page_number": idx,
                "text": cleaned_text
            })
    return pages_data

# تحميل كل ملفات الـ PDF المتاحة
all_extracted_pages = []
pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")]

print(f"📚 تم العثور على {len(pdf_files)} ملفات PDF:")
for pdf_file in pdf_files:
    full_path = os.path.join(PDF_DIR, pdf_file)
    pages = load_pdf_pages(full_path)
    all_extracted_pages.extend(pages)
    print(f"  ✓ {pdf_file}: تم استخراج {len(pages)} صفحة.")

print(f"\n📊 إجمالي الصفحات المستخرجة: {len(all_extracted_pages)} صفحة.")

## ✂️ المرحلة 3: تقطيع النصوص مع الحفاظ على الميتاداتا (Chunking)

**لماذا نقوم بالتقطيع (Chunking)؟**
نماذج الـ Embedding والـ LLM لها حدود في عدد الكلمات (Token Limits)، كما أن استرجاع مقطع دقيق بطول **500 - 600 كلمة** أفضل بكثير من استرجاع مستند كامل مكون من 40 صفحة.

**ماذا سنرفق مع كل Chunk؟**
- `content`: نص المقطع
- `metadata`: اسم المستند + رقم الصفحة + عنوان تقريبي للقسم

In [ ]:
# دالة تقسيم النصوص إلى مقاطع (Chunks) بحجم محدد مع تداخل (Overlap)
def split_text_into_chunks(pages, chunk_size=500, chunk_overlap=100):
    chunks = []
    chunk_counter = 1
    
    for page_info in pages:
        words = page_info["text"].split()
        filename = page_info["filename"]
        page_num = page_info["page_number"]
        
        # إذا كانت الصفحة قصيرة، نعتبرها Chunk واحد
        if len(words) <= chunk_size:
            chunks.append({
                "chunk_id": f"chk_{chunk_counter:04d}",
                "filename": filename,
                "page_number": page_num,
                "section": "General Clinical Overview",
                "content": " ".join(words)
            })
            chunk_counter += 1
        else:
            # تقسيم الصفحة لمقاطع متداخلة
            for i in range(0, len(words), chunk_size - chunk_overlap):
                chunk_words = words[i : i + chunk_size]
                if len(chunk_words) > 30: # تجاهل المقاطع شديدة القصر
                    chunks.append({
                        "chunk_id": f"chk_{chunk_counter:04d}",
                        "filename": filename,
                        "page_number": page_num,
                        "section": f"Section at word {i}",
                        "content": " ".join(chunk_words)
                    })
                    chunk_counter += 1
                    
    return chunks

# تنفيذ التقطيع
all_chunks = split_text_into_chunks(all_extracted_pages, chunk_size=450, chunk_overlap=80)
print(f"✅ تم تقسيم البيانات إلى {len(all_chunks)} مقطع (Chunk) جاهز للفهرسة!")

# عرض عينة لمقطع واحد وميتاداتا الخاصة به
sample = all_chunks[0]
print("\n🔍 عينة من المقطع الأول:")
print(f"• المعرف (ID): {sample['chunk_id']}")
print(f"• الملف: {sample['filename']}")
print(f"• الصفحة: {sample['page_number']}")
print(f"• بداية النص: {sample['content'][:200]}...")

## 🧠 المرحلة 4: توليد المتجهات الدلالية (Text Embeddings)

الـ **Embedding** هو تحويل الجمل والنصوص الطبية إلى أرقام (Vector من 384 بُعد) تعبر عن **المعنى والمفهوم** وليس فقط تطابق الحروف.
سنستخدم نموذج `BAAI/bge-small-en-v1.5` وهو من أقوى النماذج المتخصصة والسريعة على المعالج (CPU).

In [ ]:
# تحميل نموذج الـ Embeddings محلياً على الـ CPU
print("⏳ جاري تحميل نموذج التضمين BAAI/bge-small-en-v1.5...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
print("✅ تم تحميل النموذج بنجاح!")

# دالة مساعدة لتوليد الـ Embedding لأي نص أو قائمة نصوص
def get_embedding(texts):
    if isinstance(texts, str):
        texts = [texts]
    # normalize_embeddings=True يجعل تشابه جيب التمام (Cosine Similarity) مساوياً لضرب المصفوفات
    return embed_model.encode(texts, normalize_embeddings=True).tolist()

## 🗄️ المرحلة 5: الفهرسة في قاعدة البيانات المتجهة (ChromaDB) وبناء فهرس BM25

سننشئ نوعين من الفهارس لنحقق **البحث الهجين (Hybrid Search)**:
1. **ChromaDB**: للبحث الدلالي عن طريق متجهات المعاني (Dense Search).
2. **BM25**: للبحث السريع عن الكلمات المفتاحية وأسماء الأدوية والجينات (Lexical Keyword Search).

In [ ]:
# 1. إعداد ChromaDB (محلية في الذاكرة أو مجلد مؤقت)
chroma_client = chromadb.Client()
collection_name = "lab_clinical_guidelines"

# حذف المجموعة السابقة إن وجدت للبدء من الصفر
try:
    chroma_client.delete_collection(collection_name)
except:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

# تجهيز البيانات لـ ChromaDB
ids = [c["chunk_id"] for c in all_chunks]
documents = [c["content"] for c in all_chunks]
metadatas = [{"filename": c["filename"], "page_number": c["page_number"], "section": c["section"]} for c in all_chunks]

print("⏳ جاري حساب Embeddings لجميع المقاطع وإضافتها لـ ChromaDB...")
chunk_embeddings = get_embedding(documents)

collection.add(
    ids=ids,
    embeddings=chunk_embeddings,
    documents=documents,
    metadatas=metadatas
)
print(f"✅ تم حفظ {collection.count()} متجه في ChromaDB بنجاح!")

# 2. بناء فهرس BM25 للكلمات المفتاحية
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25_index = BM25Okapi(tokenized_corpus)
print("✅ تم بناء فهرس BM25 بنجاح!")

## 🌐 المرحلة 6: توسيع الاستعلامات الطبية (Medical Query Expansion)

إذا سأل الطبيب باللغة العربية (مثلاً: *"ما هي جرعة نوسينيرسين في ضمور العضلات الشوكي؟"*) والمستندات الطبية باللغة الإنجليزية:
نقوم بتوسيع الاستعلام بإضافة المصطلحات العلمية المقابلة (`SMA`, `Spinal Muscular Atrophy`, `Nusinersen`, `Dosing`) لضمان استرجاع أدق المقاطع.

In [ ]:
ARABIC_MEDICAL_DICT = {
    "ضمور العضلات": ["Spinal Muscular Atrophy", "SMA", "SMN1"],
    "نوسينيرسين": ["nusinersen", "Spinraza"],
    "زولجينسما": ["onasemnogene abeparvovec", "Zolgensma"],
    "ريسديبلام": ["risdiplam", "Evrysdi"],
    "جرعة": ["dosing", "dose", "loading doses"],
    "تشخيص": ["diagnosis", "carrier screening", "exome"],
    "كروموسوم": ["chromosomal rearrangements", "structural variants"]
}

def expand_medical_query(query):
    expansions = []
    for ar_term, en_terms in ARABIC_MEDICAL_DICT.items():
        if ar_term in query:
            expansions.extend(en_terms)
    
    if expansions:
        return f"{query} {' '.join(set(expansions))}"
    return query

# تجربة التوسيع
test_q = "ما هي معايير بدء علاج نوسينيرسين في ضمور العضلات؟"
print(f"السؤال الأصلي: {test_q}")
print(f"السؤال بعد التوسيع: {expand_medical_query(test_q)}")

## 🔍 المرحلة 7: محرك البحث الهجين (Hybrid Retrieval & RRF)

نجمع بين **البحث الدلالي** (ChromaDB) و **بحث الكلمات** (BM25) باستخدام خوارزمية **Reciprocal Rank Fusion (RRF)**:
$$\text{RRF Score} = \frac{\text{Weight}_{\text{Dense}}}{60 + \text{Rank}_{\text{Dense}}} + \frac{\text{Weight}_{\text{BM25}}}{60 + \text{Rank}_{\text{BM25}}}$$

In [ ]:
def hybrid_retrieve(query, top_k=4, dense_weight=0.7, bm25_weight=0.3):
    expanded_q = expand_medical_query(query)
    
    # 1. البحث الدلالي (Dense Search)
    q_emb = get_embedding(expanded_q)
    dense_res = collection.query(
        query_embeddings=q_emb,
        n_results=top_k * 2
    )
    
    # 2. بحث الكلمات (BM25)
    tokenized_q = expanded_q.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_q)
    top_bm25_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k * 2]
    
    # 3. دمج الرتب (RRF)
    rrf_scores = {}
    chunk_lookup = {}
    
    # درجات البحث الدلالي
    for rank, (cid, doc, meta, dist) in enumerate(zip(
        dense_res["ids"][0],
        dense_res["documents"][0],
        dense_res["metadatas"][0],
        dense_res["distances"][0]
    )):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (dense_weight / (60 + rank + 1))
        sim_score = max(0.0, 1.0 - dist) # تحويل مسافة Cosine لتشابه
        chunk_lookup[cid] = {"id": cid, "content": doc, "metadata": meta, "similarity": round(sim_score, 3)}
        
    # درجات BM25
    for rank, idx in enumerate(top_bm25_idx):
        cid = all_chunks[idx]["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (bm25_weight / (60 + rank + 1))
        if cid not in chunk_lookup:
            chunk_lookup[cid] = {
                "id": cid,
                "content": all_chunks[idx]["content"],
                "metadata": {
                    "filename": all_chunks[idx]["filename"],
                    "page_number": all_chunks[idx]["page_number"],
                    "section": all_chunks[idx]["section"]
                },
                "similarity": 0.70
            }
            
    # ترتيب النتائج النهائية
    sorted_cids = sorted(rrf_scores.keys(), key=lambda k: rrf_scores[k], reverse=True)
    final_results = [chunk_lookup[cid] for cid in sorted_cids[:top_k]]
    return final_results

# تجربة استرجاع
results = hybrid_retrieve("معايير بدء علاج النوسينيرسين في أطفال SMA", top_k=3)
print(f"🎯 تم استرجاع {len(results)} مقاطع دقيقة:")
for r in results:
    print(f"• [{r['metadata']['filename']} | صفحة {r['metadata']['page_number']}] (درجة التشابه: {r['similarity']})")

## 🛡️ المرحلة 8: بوابات الأمان السريري وتدقيق الهلوسة

في الأنظمة الطبية، إذا كانت درجة التشابه ضعيفة (أقل من `0.60`) يجب أن **يرفض النظام الإجابة بأمان** بدلاً من التخمين.

In [ ]:
def evaluate_safety_gate(retrieved_chunks, threshold=0.60):
    if not retrieved_chunks:
        return False, 0.0, "لم يتم العثور على أي أدلة سريرية مرتبطة."
    
    max_similarity = max(c["similarity"] for c in retrieved_chunks)
    if max_similarity < threshold:
        return False, max_similarity, f"درجة مطابقة الأدلة ({max_similarity:.2f}) أقل من حد الأمان المطلوب ({threshold})."
    
    return True, max_similarity, "تم اجتياز بوابة الأمان بنجاح - الأدلة كافية وموثوقة."

## ✍️ المرحلة 9: التوليد المنضبط بالأدلة (Grounded Generation) مع الاستشهادات

نقوم ببناء Prompt يلزم النموذج بالإجابة **فقط من واقع المقاطع المسترجعة** مع ذكر الاستشهاد `[الملف | صفحة X]` لكل معلومة.

In [ ]:
# دالة صياغة الـ Prompt الطبي المنضبط
def build_clinical_prompt(query, chunks, language="ar"):
    context_text = ""
    for idx, c in enumerate(chunks, 1):
        meta = c["metadata"]
        context_text += f"\n[PASSAGE #{idx} | Source: {meta['filename']} | Page: {meta['page_number']}]\n{c['content']}\n"
        
    lang_instruction = (
        "يرجى تقديم التوصية السريرية باللغة العربية الواضحة، مع الاحتفاظ بأسماء الأدوية والجينات باللغة الإنجليزية."
        if language == "ar" else
        "Provide your clinical recommendation in clear professional English."
    )
    
    prompt = f"""أنت VERA، مساعد سريري ذكي للأطباء. أجب حصرياً وبدقة تامة بناءً على الأدلة الطبية المسترجعة التالية:
### الأدلة الطبية المسترجعة:
{context_text}

### استفسار الطبيب:
{query}

### تعليمات الإجابة:
1. قدم ملخصاً سريرياً مباشراً (Executive Summary).
2. اذكر التوصيات والجرعات في نقاط واضحة مع ذكر الاستشهاد بجانب كل معلومة بالشكل: [{chunks[0]['metadata']['filename']} | Page X].
3. {lang_instruction}
4. لا تختلق أي معلومة غير موجودة في السياق.
"""
    return prompt

# دالة التوليد مع دعم Google Gemini أو وضع تجريبي محلي بدون مفتاح
def generate_clinical_answer(prompt, chunks, api_key=None):
    key = api_key or os.getenv("GEMINI_API_KEY")
    
    # إذا كان المفتاح متوفراً، نستخدم Gemini
    if key:
        try:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model = genai.GenerativeModel("models/gemini-3.7-flash")
            response = model.generate_content(prompt)
            if response and response.text:
                return response.text
        except Exception as e:
            print(f"⚠️ حدث خطأ في Gemini API ({e})، سيتم استخدام التوليد التجريبي المحلي.")
            
    # وضع المحاكاة المحلي (Deterministic Fallback) في حال عدم توفر مفتاح API
    top_chunk = chunks[0]
    meta = top_chunk["metadata"]
    return f"""### ملخص التوصية السريرية:
بناءً على الأدلة الإكلينيكية المسترجعة من [{meta['filename']} | صفحة {meta['page_number']}]، تشير الإرشادات إلى:
• {top_chunk['content'][:250]}...

### الاستشهادات المرجعية:
• المصدر: {meta['filename']}
• رقم الصفحة: صفحة {meta['page_number']}"""

## 🧪 المرحلة 10: المختبر التفاعلي الشامل (End-to-End Clinical Query Tester)

اكتب أي استفسار طبي أدناه لتشاهد منظومة **VERA** تعمل بكامل مراحلها أمامك!

In [ ]:
# 🩺 ضع سؤالك الطبي هنا لتجربة المنظومة بالكامل:
MY_CLINICAL_QUERY = "ما هي معايير بدء علاج النوسينيرسين (Nusinersen) وجرعات التحميل لمرضى SMA؟"
LANGUAGE = "ar" # 'ar' أو 'en'
OPTIONAL_GEMINI_API_KEY = "" # يمكنك وضع مفتاحك هنا إذا أردت أو تركه فارغاً

print("=" * 75)
print(f"🩺 الاستعلام السريري: {MY_CLINICAL_QUERY}")
print("=" * 75)

# الخطوة 1: الاسترجاع الهجين
retrieved = hybrid_retrieve(MY_CLINICAL_QUERY, top_k=3)
print(f"\n🔍 [الخطوة 1] تم استرجاع {len(retrieved)} مقاطع من الأوراق العلمية:")
for idx, r in enumerate(retrieved, 1):
    print(f"   {idx}. {r['metadata']['filename']} (صفحة {r['metadata']['page_number']}) - تشابه: {r['similarity']}")

# الخطوة 2: فحص بوابة الأمان
passed, score, msg = evaluate_safety_gate(retrieved)
print(f"\n🛡️ [الخطوة 2] بوابة الأمان: {'✅ اجتازت' if passed else '❌ رُفضت'} ({msg})")

# الخطوة 3: التوليد المنضبط بالأدلة
if passed:
    prompt = build_clinical_prompt(MY_CLINICAL_QUERY, retrieved, language=LANGUAGE)
    print("\n✍️ [الخطوة 3] جاري صياغة الإجابة الموثقة بالاستشهادات...")
    clinical_answer = generate_clinical_answer(prompt, retrieved, api_key=OPTIONAL_GEMINI_API_KEY)
    
    print("\n" + "-" * 75)
    print(clinical_answer)
    print("-" * 75)
    print("\n⚠️ إخلاء مسؤولية: هذا النظام للمساعدة البحثية ولا يحل محل القرار الطبي المباشر.")
else:
    print("\n⛔ تم حجب الإجابة لعدم توفر أدلة سريرية كافية مطابقة لمعايير الأمان.")

In [4]:
from google import genai

client = genai.Client(api_key="YOUR_GEMINI_API_KEY_HERE")

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain how AI works in a few words"
)
print(interaction.output_text)

ImportError: cannot import name 'genai' from 'google' (unknown location)

In [2]:
!pip install google